In [1]:
import sys
print(sys.executable)


/Users/tanishaprabhu/Desktop/analytrix/Pandas-and-Co/.venv/bin/python


In [2]:
import pandas as pd
import numpy as np
import json
import re
import base64

In [3]:
path = "../data/raw/raw_data.csv"
df = pd.read_csv(path)

df.head()

,ID,Age,Education_Level,Years_of_Experience,Current_or_Target_Role,Technical_Skill_Score,Soft_Skill_Score,Domain_Knowledge_Score,Certification_Count,Training_Hours_Last_Year,Skill_Gap_Score,Readiness_Score,Employment_Status,Industry,Annual_Salary_USD,Career_Readiness_Level
0,0x1,NaN,Diploma,0.6,Cybersecurity Analyst,61,51-56,65,0,310,41.87,47.35,Unemployed,NaN,118274 USD,Medium
1,2,NaN,NaN,8.4,?,"{""v"": 59}",MISSING,MISSING,2,"{""v"": 175}",50.91,"{""v"": 79.52}",?,MISSING,32296,High
2,3,NaN,Bachelor,7.3-12.3,tsylanA ataD,48,"{""v"": 57}",92,7,35,30.15,47.31,Unemployed,UmV0YWls,26393,Low
3,4,"{""v"": 25}",Diploma,11.2,NaN,49,42-47,"{""v"": 94}",2,301,"{""v"": 42.46}",87.86-92.86,NaN,Finance,20121,High
4,0x5,38,MISSING,0.4,NaN,NaN,66,66,?,291,90.29-95.29,MISSING,deyolpmE,Healthcare,25655,TG93


In [4]:
df.shape

(50000, 16)

In [5]:
df.columns

Index(['ID', 'Age', 'Education_Level', 'Years_of_Experience',
       'Current_or_Target_Role', 'Technical_Skill_Score', 'Soft_Skill_Score',
       'Domain_Knowledge_Score', 'Certification_Count',
       'Training_Hours_Last_Year', 'Skill_Gap_Score', 'Readiness_Score',
       'Employment_Status', 'Industry', 'Annual_Salary_USD',
       'Career_Readiness_Level'],
      dtype='object')

In [6]:
df.isna().sum().sort_values(ascending=False).head(10)

Age                         6093
Years_of_Experience         6077
Readiness_Score             6070
Industry                    6061
Certification_Count         6060
Soft_Skill_Score            6046
Training_Hours_Last_Year    6021
Education_Level             6016
Career_Readiness_Level      6009
Current_or_Target_Role      6008
dtype: int64

In [7]:
def extract_value(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, str):
        x = x.strip()

        # common missing markers
        if x in {"?", "MISSING", "", "NaN"}:
            return np.nan

        # JSON-like entries - {"v" : number}
        if x.startswith("{") and "v" in x:
            try:
                # turn text into python dict {"key" : value}
                val = json.loads(x.replace("'", '"')).get("v")
                return val
            except:
                pass

        # Range pattern: "a-b" -> mean
        if re.match(r"^\d+(\.\d+)?-\d+(\.\d+)?$", x):
            a, b = map(float, x.split("-"))
            return (a + b) / 2

        # numeric embedded in text: "123 USD" -> 123
        num = re.findall(r"\d+\.?\d*", x)
        if num:
            return float(num[0])

    return x


In [8]:
def base64_decode(val):
    if not isinstance(val,str):
        return val
    try:
        decoded = base64.b64decode(val).decode("utf-8")
        # if decoded text looks like normal text, we keep it
        if all(31 < ord(c) < 127 for c in decoded):
            return decoded
    except:
        pass
    return val

def reverse_val(val):
    if not isinstance(val, str):
        return val

    words = val.split()

    sus = sum(
        1 for w in words if len(w) > 3 and w[-1].isupper()
    )
    if sus >= len(words) / 2:
        return val[::-1]

    return val

    

In [9]:
for col in df.columns:
    df[col] = df[col].apply(extract_value)

text_cols = [
    "Current_or_Target_Role",
    "Employment_Status",
    "Industry",
    "Career_Readiness_Level"
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].apply(reverse_val)
        df[col] = df[col].apply(base64_decode)

        
df[text_cols].head()

,Current_or_Target_Role,Employment_Status,Industry,Career_Readiness_Level
0,Cybersecurity Analyst,Unemployed,NaN,Medium
1,NaN,NaN,NaN,High
2,Data Analyst,Unemployed,0.0,Low
3,NaN,NaN,Finance,High
4,NaN,Employed,Healthcare,93.0


In [13]:
numeric_cols = [
    "Age",
    "Years_of_Experience",
    "Technical_Skill_Score",
    "Soft_Skill_Score",
    "Domain_Knowledge_Score",
    "Certification_Count",
    "Training_Hours_Last_Year",
    "Skill_Gap_Score",
    "Readiness_Score",
    "Annual_Salary_USD",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col])

In [14]:
df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,39950.0,38.821990,12.101495,18.00,28.50,39.00,49.0000,61.50
Years_of_Experience,39903.0,5.252605,5.036816,0.00,1.60,3.80,7.2000,47.00
Technical_Skill_Score,40198.0,69.847368,17.304886,40.00,55.00,70.00,85.0000,101.50
Soft_Skill_Score,39967.0,69.693635,17.331058,40.00,55.00,70.00,85.0000,101.50
Domain_Knowledge_Score,40240.0,69.695403,17.319549,40.00,55.00,70.00,85.0000,101.50
Certification_Count,39802.0,3.749359,2.407890,0.00,2.00,4.00,6.0000,9.50
Training_Hours_Last_Year,40033.0,250.792883,143.977866,0.00,127.00,252.00,375.0000,501.50
Skill_Gap_Score,40036.0,50.022880,28.843955,0.00,24.90,49.66,75.0125,102.50
Readiness_Score,39775.0,49.964836,28.956701,0.01,24.94,49.81,75.0600,102.42
Annual_Salary_USD,40017.0,85105.434165,37622.452439,20000.00,52179.00,84999.00,117750.0000,149997.00


In [15]:
#  Handle missing values
# numeric fill with median
present_numeric = [c for c in numeric_cols if c in df.columns]
df[present_numeric] = df[present_numeric].fillna(df[present_numeric].median())

# categorical fill with Unknown
cat_cols = [c for c in (text_cols + ["Education_Level"]) if c in df.columns]
for c in cat_cols:
    df[c] = df[c].fillna("Unknown")

df.isna().sum().sort_values(ascending=False).head(10)


ID                          0
Age                         0
Education_Level             0
Years_of_Experience         0
Current_or_Target_Role      0
Technical_Skill_Score       0
Soft_Skill_Score            0
Domain_Knowledge_Score      0
Certification_Count         0
Training_Hours_Last_Year    0
dtype: int64

In [16]:
# Validation Checks
df.info()

if "ID" in df.columns:
    print("Duplicate IDs:", df["ID"].duplicated().sum())

if "Career_Readiness_Level" in df.columns:
    print(df["Career_Readiness_Level"].value_counts().head(20))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        50000 non-null  float64
 1   Age                       50000 non-null  float64
 2   Education_Level           50000 non-null  object 
 3   Years_of_Experience       50000 non-null  float64
 4   Current_or_Target_Role    50000 non-null  object 
 5   Technical_Skill_Score     50000 non-null  float64
 6   Soft_Skill_Score          50000 non-null  float64
 7   Domain_Knowledge_Score    50000 non-null  float64
 8   Certification_Count       50000 non-null  float64
 9   Training_Hours_Last_Year  50000 non-null  float64
 10  Skill_Gap_Score           50000 non-null  float64
 11  Readiness_Score           50000 non-null  float64
 12  Employment_Status         50000 non-null  object 
 13  Industry                  50000 non-null  object 
 14  Annual

In [22]:
#  For Career Readiness Level fixes:
valid_levels = {"Low", "Medium", "High", "Unknown"}

def fix_reversed_level(val):
    if not isinstance(val, str):
        return val

    rev = val[::-1]

    if rev in valid_levels:
        return rev

    return val

def numeric_to_level(row):
    val = row["Career_Readiness_Level"]

    if isinstance(val, (int, float)):
        score = row["Readiness_Score"]

        if score < 40:
            return "Low"
        elif score < 70:
            return "Medium"
        else:
            return "High"

    return val

df["Career_Readiness_Level"] = df.apply(numeric_to_level, axis=1)


df["Career_Readiness_Level"] = (
    df["Career_Readiness_Level"]
    .apply(fix_reversed_level)
)

In [23]:
df['Career_Readiness_Level'].head(20)


0      Medium
1        High
2         Low
3        High
4      Medium
5        High
6      Medium
7      Medium
8        High
9         Low
10     Medium
11     Medium
12       High
13     Medium
14        Low
15       High
16     Medium
17        Low
18        Low
19    Unknown
Name: Career_Readiness_Level, dtype: object

In [24]:
df["Career_Readiness_Level"].value_counts().head(20)

Career_Readiness_Level
Medium     15223
High       14268
Low        10455
Unknown    10054
Name: count, dtype: int64

In [25]:
clean_path = "../data/processed/clean_data.csv"
df.to_csv(clean_path, index=False)
clean_path


'../data/processed/clean_data.csv'